# Construire le classeur Excel « jouable » (eSAF, TYNDP 2026)

Ce carnet fabrique `esaf_trajectoire_hypotheses.xlsx` à la racine du dépôt. Le classeur reprend le calcul du carnet
`esaf_trajectoire.ipynb` (UE-27), mais **avec des formules Excel** : changez une hypothèse (cellules jaunes de la feuille
`Hypothèses`) et tout le classeur se recalcule, graphique compris.

- Données : extraits du dossier `data/` (*TYNDP 2026 Scenarios*, ENTSO-E / ENTSOG, CC BY 4.0 ; ReFuelEU annexe I).
- Seule la feuille `Données` contient des nombres écrits par Python (sommes UE-27). Tout le reste est formule.
- Bibliothèque : `openpyxl` (écrit des fichiers .xlsx ; elle n'évalue pas les formules : c'est Excel ou LibreOffice qui le fait à l'ouverture).

In [1]:
import sys
from pathlib import Path
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import LineChart, Reference
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.worksheet.datavalidation import DataValidation

RACINE = next(p for p in [Path("."), Path("..")] if (p / "data" / "countries.csv").exists())
DATA = RACINE / "data"
SORTIE = RACINE / "esaf_trajectoire_hypotheses.xlsx"
print("Données :", DATA.resolve(), "| sortie :", SORTIE.resolve())

Données : /Users/simonbrigode/paper_projects/tyndp-esaf-students/data | sortie : /Users/simonbrigode/paper_projects/tyndp-esaf-students/esaf_trajectoire_hypotheses.xlsx


## 1. Les données UE-27 (somme des 27 pays, comme dans le carnet principal)

In [2]:
pays = pd.read_csv(DATA / "countries.csv")
UE27 = pays.loc[pays.ue27 == "oui", "code"].tolist()
kpi = pd.read_csv(DATA / "tyndp_efuels_kpi.csv")
liq = pd.read_csv(DATA / "liquids_all_sectors.csv")
av = pd.read_csv(DATA / "aviation_fuel_by_carrier.csv")
ref = pd.read_csv(DATA / "refueleu_annex1.csv")

k = kpi[kpi.country == "EU27"]
def ligne(label, section=None):
    s = k[k.label == label]
    if section:
        s = s[s.section == section]
    return s.set_index(["variant", "year"]).value

D = pd.DataFrame({
    "h2_vers_eliquides": ligne("H2 to E-Liquids", "Consumption"),
    "imports_eliquides": ligne("E-Liquid Imports [TWh H2]"),
})
D["liquides_tous_secteurs"] = liq[liq.country.isin(UE27)].groupby(["variant", "year"]).twh.sum()
D["aviation_liquides"] = av[av.country.isin(UE27) & (av.carrier == "Liquids")].groupby(["variant", "year"]).twh.sum()
D = D.dropna().reset_index()
D

,variant,year,h2_vers_eliquides,imports_eliquides,liquides_tous_secteurs,aviation_liquides
0,HEV,2035,118.506436,153.610087,3876.879663,589.309528
1,HEV,2040,142.938755,324.323850,3038.130142,600.657775
2,LEV,2035,123.647243,148.469279,4115.233954,577.507843
3,LEV,2040,203.467166,263.795439,3414.608811,581.722519
4,NT+,2030,43.499614,64.449377,4840.274307,603.889801
5,NT+,2035,122.915366,149.201157,3995.596188,583.356281
6,NT+,2040,187.722426,279.540179,3226.177669,591.052906
7,NT+,2050,410.933949,481.668529,1964.426906,566.403039


## 2. Le classeur
Feuilles : `Lisez-moi`, `Hypothèses` (cellules jaunes modifiables), `Trajectoire`, `Besoins`, `Données`, `ReFuelEU`.

In [3]:
JAUNE = PatternFill("solid", fgColor="FFF2CC")
GRIS = PatternFill("solid", fgColor="EDEDED")
GRAS = Font(bold=True)
TITRE = Font(bold=True, size=14)

wb = Workbook()
lm = wb.active
lm.title = "Lisez-moi"
textes = [
    ("Trajectoire eSAF dans le TYNDP 2026 : classeur à hypothèses modifiables", TITRE),
    ("", None),
    ("1. Feuille « Hypothèses » : modifiez UNIQUEMENT les cellules jaunes (scénario, rendements, règles, variations).", None),
    ("2. Les feuilles « Trajectoire » et « Besoins » se recalculent seules (formules) ; le graphique suit.", None),
    ("3. Feuilles « Données » et « ReFuelEU » : valeurs sources (ne pas modifier, sauf exercice).", None),
    ("", None),
    ("Méthode (identique au carnet esaf_trajectoire.ipynb) :", GRAS),
    ("  e-liquides (TWh de carburant) = (H2 vers e-liquides + imports e-liquides, en TWh H2) × rendement H2 → liquide", None),
    ("  eSAF au prorata = liquides de l'aviation × e-liquides / liquides tous secteurs ; « aviation d'abord » = min(e-liquides, liquides aviation)", None),
    ("  Obligation ReFuelEU (sous-obligation synthétique, % de l'énergie) × liquides de l'aviation = volume exigé", None),
    ("  Besoins amont : H2 = eSAF / η(H2→liquide) ; électricité = eSAF / η(élec→carburant) ; CO2 = eSAF × g/MJ × 3,6 ; eau = H2 × kg/kWh", None),
    ("", None),
    ("Limites : TYNDP ne distingue pas kérosène fossile, bio et synthétique (un seul vecteur « Liquids ») : l'eSAF est DÉRIVÉ.", None),
    ("Périmètre : UE-27 (somme des 27 pays). Années : NT+ 2030, 2035, 2040, 2050 ; HEV et LEV 2035, 2040 seulement (cellules vides sinon).", None),
    ("", None),
    ("Sources : « TYNDP 2026 Scenarios », ENTSO-E / ENTSOG, licence CC BY 4.0 (extraits remis en forme, non endossés par les ENTSO) ;", None),
    ("Règlement (UE) 2023/2405 (ReFuelEU Aviation), annexe I. Détail : data/SOURCES.md. Classeur généré par notebooks/construire_excel.ipynb.", None),
]
for i, (t, f) in enumerate(textes, start=1):
    c = lm.cell(row=i, column=1, value=t)
    if f:
        c.font = f
lm.column_dimensions["A"].width = 140

In [4]:
hy = wb.create_sheet("Hypothèses")
hy["A1"] = "Hypothèses (cellules jaunes modifiables)"; hy["A1"].font = TITRE
hy.append([])
hy.append(["Hypothèse", "Valeur", "Unité", "Repère / source"])
for c in hy[3]:
    c.font = GRAS
HYP = [  # (nom défini, libellé, valeur, unité, commentaire)
    ("Scenario", "Scénario TYNDP", "NT+", "", "NT+ (National Trends+), HEV ou LEV (variantes 2035-2040)"),
    ("Eta_H2_liq", "Rendement H2 → carburant liquide", round(0.51 / 0.75, 3), "TWh carb. / TWh H2", "Central = 0,51 / 0,75 (Concawe 4/24) ; fourchette du carnet 0,56 à 0,70"),
    ("Eta_elec_fuel", "Rendement électricité → carburant", 0.51, "TWh carb. / TWh élec.", "Central 0,51 (Concawe 4/24, 2050) ; fourchette 0,42 à 0,60"),
    ("CO2_g_MJ", "CO2 lié dans le carburant", 73.6, "g CO2 / MJ", "Concawe 4/24 ; stœchiométrie CH2 ≈ 72,7"),
    ("Eau_kg_kWh", "Eau d'électrolyse (plancher)", round(9 / 33.33, 4), "kg eau / kWh H2", "9 kg d'eau par kg d'H2 (PCI 33,33 kWh/kg) ; sans refroidissement"),
    ("Regle", "Règle d'affectation des e-liquides à l'aviation", "prorata", "", "prorata ou aviation d'abord"),
    ("Volume_besoins", "Volume d'eSAF pour la feuille Besoins", "obligation", "", "obligation (ReFuelEU) ou TYNDP (eSAF dérivé)"),
    ("Var_aviation", "Variation de la demande de liquides de l'aviation", 0.0, "%", "0 = TYNDP tel quel ; ex. -20 = sobriété, +20 = croissance"),
    ("Mult_obligation", "Ambition de l'obligation synthétique", 100.0, "% de ReFuelEU", "100 = ReFuelEU tel quel ; 150 = 1,5 fois plus exigeant"),
]
for i, (nom, lib, val, unite, com) in enumerate(HYP, start=4):
    hy.cell(row=i, column=1, value=lib)
    c = hy.cell(row=i, column=2, value=val); c.fill = JAUNE
    hy.cell(row=i, column=3, value=unite)
    hy.cell(row=i, column=4, value=com)
    wb.defined_names[nom] = DefinedName(nom, attr_text=f"'Hypothèses'!$B${i}")
for col, w in zip("ABCD", (46, 14, 22, 80)):
    hy.column_dimensions[col].width = w
lignes = {nom: i for i, (nom, *_r) in enumerate(HYP, start=4)}
for nom, liste in [("Scenario", '"NT+,HEV,LEV"'), ("Regle", '"prorata,aviation d\'abord"'), ("Volume_besoins", '"obligation,TYNDP"')]:
    dv = DataValidation(type="list", formula1=liste, allow_blank=False)
    hy.add_data_validation(dv); dv.add(f"B{lignes[nom]}")

In [5]:
do = wb.create_sheet("Données")
do.append(["variant", "annee", "H2 vers e-liquides (TWh H2)", "Imports e-liquides (TWh H2)",
           "Liquides tous secteurs (TWh)", "Liquides aviation (TWh)"])
for c in do[1]:
    c.font = GRAS
for r in D.itertuples(index=False):
    do.append([r.variant, int(r.year), round(r.h2_vers_eliquides, 6), round(r.imports_eliquides, 6),
               round(r.liquides_tous_secteurs, 6), round(r.aviation_liquides, 6)])
for col in "ABCDEF":
    do.column_dimensions[col].width = 26
nD = do.max_row

rf = wb.create_sheet("ReFuelEU")
rf.append(["annee", "SAF minimum (%)", "dont synthétique minimum (%)", "remarque"])
for c in rf[1]:
    c.font = GRAS
for r in ref.itertuples(index=False):
    rf.append([int(r.annee), float(r.part_saf_min_pct),
               0.0 if pd.isna(r.part_synthetique_min_pct) else float(r.part_synthetique_min_pct), r.remarque])
rf.column_dimensions["D"].width = 110
nR = rf.max_row

In [6]:
tr = wb.create_sheet("Trajectoire", 2)
tr["A1"] = "Trajectoire eSAF, UE-27 (recalculée à partir des hypothèses)"; tr["A1"].font = TITRE
tr["A2"] = '="Scénario : "&Scenario&" ; règle : "&Regle&" ; η H2→liquide = "&TEXT(Eta_H2_liq,"0.000")'
ENTETES = ["Année", "H2 vers e-liquides (TWh H2)", "Imports e-liquides (TWh H2)", "e-liquides (TWh carb.)",
           "Liquides tous secteurs (TWh)", "Liquides aviation (TWh)", "Part des e-liquides dans les liquides (%)",
           "eSAF (TWh)", "Part d'eSAF dans l'aviation (%)", "Obligation synthétique (%)", "Obligation (TWh)",
           "Écart eSAF − obligation (TWh)"]
tr.append([]); tr.append(ENTETES)
for c in tr[4]:
    c.font = GRAS; c.alignment = Alignment(wrap_text=True, vertical="top")
ANNEES = [2030, 2035, 2040, 2050]
def sumifs(col, r):
    return f"SUMIFS('Données'!${col}$2:${col}${nD},'Données'!$A$2:$A${nD},Scenario,'Données'!$B$2:$B${nD},$A{r})"
for j, an in enumerate(ANNEES):
    r = 5 + j
    ok = f"COUNTIFS('Données'!$A$2:$A${nD},Scenario,'Données'!$B$2:$B${nD},$A{r})>0"
    f = {
        "A": an,
        "B": f'=IF({ok},{sumifs("C", r)},"")',
        "C": f'=IF({ok},{sumifs("D", r)},"")',
        "D": f'=IF(B{r}="","",(B{r}+C{r})*Eta_H2_liq)',
        "E": f'=IF({ok},{sumifs("E", r)},"")',
        "F": f'=IF({ok},{sumifs("F", r)}*(1+Var_aviation/100),"")',
        "G": f'=IF(D{r}="","",100*D{r}/E{r})',
        "H": f'=IF(D{r}="","",IF(Regle="prorata",F{r}*D{r}/E{r},MIN(D{r},F{r})))',
        "I": f'=IF(H{r}="","",100*H{r}/F{r})',
        "J": f"=LOOKUP($A{r},'ReFuelEU'!$A$2:$A${nR},'ReFuelEU'!$C$2:$C${nR})*Mult_obligation/100",
        "K": f'=IF(F{r}="","",J{r}/100*F{r})',
        "L": f'=IF(H{r}="","",H{r}-K{r})',
    }
    for col, v in f.items():
        tr[f"{col}{r}"] = v
        if col != "A":
            tr[f"{col}{r}"].number_format = "0.0"
for col in "ABCDEFGHIJKL":
    tr.column_dimensions[col].width = 15 if col != "A" else 8
tr.row_dimensions[4].height = 60

ch = LineChart()
ch.title = "Part d'eSAF dans les carburants de l'aviation vs obligation (UE-27)"
ch.y_axis.title = "% de l'énergie"; ch.x_axis.title = "Année"; ch.height, ch.width = 8, 18
ch.add_data(Reference(tr, min_col=9, min_row=4, max_row=8), titles_from_data=True)
ch.add_data(Reference(tr, min_col=10, min_row=4, max_row=8), titles_from_data=True)
ch.set_categories(Reference(tr, min_col=1, min_row=5, max_row=8))
tr.add_chart(ch, "B11")

In [7]:
be = wb.create_sheet("Besoins", 3)
be["A1"] = "Besoins amont du volume d'eSAF choisi (UE-27)"; be["A1"].font = TITRE
be["A2"] = '="Volume : "&Volume_besoins&" ; η H2→liquide = "&TEXT(Eta_H2_liq,"0.000")&" ; η élec→carburant = "&TEXT(Eta_elec_fuel,"0.00")'
be.append([]); be.append(["Année", "eSAF (TWh)", "H2 (TWh)", "Électricité (TWh)", "CO2 capté (Mt)", "Eau (Mm3)"])
for c in be[4]:
    c.font = GRAS
for j, an in enumerate(ANNEES):
    r = 5 + j
    be[f"A{r}"] = an
    be[f"B{r}"] = f'=IF(Volume_besoins="obligation",Trajectoire!K{r},Trajectoire!H{r})'
    be[f"C{r}"] = f'=IF(B{r}="","",B{r}/Eta_H2_liq)'
    be[f"D{r}"] = f'=IF(B{r}="","",B{r}/Eta_elec_fuel)'
    be[f"E{r}"] = f'=IF(B{r}="","",B{r}*CO2_g_MJ*3.6/1000)'
    be[f"F{r}"] = f'=IF(C{r}="","",C{r}*Eau_kg_kWh)'
    for col in "BCDEF":
        be[f"{col}{r}"].number_format = "0.0"
be["A10"] = "Unités : 1 TWh = 3,6 × 10^9 MJ ; CO2 (Mt) = TWh × g/MJ × 3,6 / 1000 ; eau (Mm3) = TWh H2 × kg/kWh (1 Mm3 = 10^9 kg)."
for col in "ABCDEF":
    be.column_dimensions[col].width = 18

for ws in (do, rf):
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.fill = GRIS
wb.save(SORTIE)
print("Écrit :", SORTIE.resolve())

Écrit : /Users/simonbrigode/paper_projects/tyndp-esaf-students/esaf_trajectoire_hypotheses.xlsx


## 3. Contrôle : les valeurs par défaut du classeur = celles du carnet principal
On recalcule en Python, avec les hypothèses par défaut, ce que les formules donneront à l'ouverture (NT+, prorata).

In [8]:
eta = round(0.51 / 0.75, 3)
c = D[D.variant == "NT+"].set_index("year")
eliq = (c.h2_vers_eliquides + c.imports_eliquides) * eta
esaf = c.aviation_liquides * eliq / c.liquides_tous_secteurs
synth = ref.set_index("annee").part_synthetique_min_pct.reindex([2030, 2035, 2040, 2050])
synth.index.name = "year"
attendu = pd.DataFrame({"e-liquides": eliq, "eSAF": esaf, "part eSAF %": 100 * esaf / c.aviation_liquides,
                        "obligation %": synth.values, "obligation TWh": synth.values / 100 * c.aviation_liquides})
attendu.round(1)

,e-liquides,eSAF,part eSAF %,obligation %,obligation TWh
year,,,,,
2030,73.4,9.2,1.5,1.2,7.2
2035,185.0,27.0,4.6,5.0,29.2
2040,317.7,58.2,9.8,10.0,59.1
2050,607.0,175.0,30.9,35.0,198.2
